# 38 Targeted ScienceWorld recovery diagnostic
CPU only. Reads the saved 18-case pilot and preserves originals. Tests obvious one-call inverse actions where available; does not perform exhaustive recovery search or assign final labels.
Run all cells in order. This notebook checks exact exported-state differences, action bindings and prior search coverage. Time-matched comparisons are diagnostic only. Full simulator-state coverage remains unverified.


In [ ]:
import sys, subprocess, os
os.environ['JAVA_TOOL_OPTIONS'] = '-Xlog:os+container=off'
subprocess.run(['java', '-version'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install',
 'git+https://github.com/allenai/ScienceWorld.git@e8216d6044e8e39be9fcb185e3b2dfb602584b52'], check=True)
print('Pinned ScienceWorld installed. No model packages needed.')


In [ ]:
from google.colab import drive
from pathlib import Path
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/ADNN_Irreversibility')
assert PROJECT.is_dir(), 'Project folder not found on Drive.'
print('Project:', PROJECT)


In [ ]:
from pathlib import Path
Path("/content/38_scienceworld_recovery_diagnostic_colab.py").write_text('"""Read pilot37 and test targeted recoveries. Never emits final training labels."""\nimport argparse, gzip, hashlib, importlib.metadata, json, os, time, traceback\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nfrom collections import Counter\nCOMMIT=\'e8216d6044e8e39be9fcb185e3b2dfb602584b52\'\nJAR=\'e77b0fee7d68abe3ca5b12e57d86e2bfe7603f20c5200da0b0f085939faeb465\'\nSOURCE=\'scienceworld_physical_pilot_20261107T091929052223Z\'\n\ndef save(p,v):\n    tmp=p.with_suffix(p.suffix+\'.tmp\');tmp.write_text(json.dumps(v,indent=2,allow_nan=False));tmp.replace(p)\ndef sha(p):return hashlib.sha256(p.read_bytes()).hexdigest()\ndef world(s):return {k:s[k] for k in (\'tree\',\'look\',\'inventory\',\'valid_actions\')}\ndef diff(a,b,path=\'$\'):\n    if type(a)!=type(b):return [{\'path\':path,\'before\':a,\'after\':b}]\n    if isinstance(a,dict):\n        result=[]\n        for k in sorted(a.keys()|b.keys()):\n            if k not in a or k not in b:result.append({\'path\':path+\'/\'+str(k),\'before\':a.get(k),\'after\':b.get(k),\'missing_key\':True})\n            else:result.extend(diff(a[k],b[k],path+\'/\'+str(k)))\n        return result\n    if isinstance(a,list):\n        if a==b:return []\n        return [{\'path\':path,\'before\':a,\'after\':b}]\n    return [] if a==b else [{\'path\':path,\'before\':a,\'after\':b}]\ndef objects(tree):\n    out={}\n    def walk(node,parent=None):\n        if not isinstance(node,dict):return\n        uid=str(node.get(\'uuid\',\'\'))\n        if uid:out[uid]={\'name\':node.get(\'name\',\'\'),\'parent\':parent}\n        contents=node.get(\'contents\',{})\n        for child in (contents.values() if isinstance(contents,dict) else contents if isinstance(contents,list) else []):walk(child,uid or parent)\n    walk(tree);return out\n\ndef suggestions(action,before,after):\n    valid=set(after[\'valid_actions\']); options=[]\n    for a,b in [(\'activate \',\'deactivate \'),(\'deactivate \',\'activate \')]:\n        if action.startswith(a):options.append(b+action[len(a):])\n    if action.startswith(\'move \'):\n        old,new=objects(before[\'tree\']),objects(after[\'tree\'])\n        for uid,item in old.items():\n            if uid in new and item[\'parent\']!=new[uid][\'parent\'] and item[\'parent\'] in old:\n                options.append(\'move \'+item[\'name\']+\' to \'+old[item[\'parent\']][\'name\'])\n    return [[x] for x in dict.fromkeys(options) if x in valid][:3]\n\ndef comparison(a,b):\n    changes=diff(world(a),world(b))\n    temps=[x for x in changes if x[\'path\'].endswith(\'/temperatureC\')]\n    return {\'exact_exported_world_match\':not changes,\'difference_count\':len(changes),\n            \'temperature_field_differences\':len(temps),\n            \'other_difference_count\':len(changes)-len(temps),\'differences\':changes,\n            \'note\':\'Temperature-only differences are not ignored or classified as reversible.\'}\n\ndef main(args):\n    source=Path(args.source);assert (source/\'results.json\').exists(),\'Pilot folder missing. Mount Drive and check source path.\'\n    config=json.loads((source/\'config.json\').read_text())\n    assert config[\'commit\']==COMMIT and config[\'jar_sha256\']==JAR\n    option=\'-Xlog:os+container=off\'\n    if option not in os.environ.get(\'JAVA_TOOL_OPTIONS\',\'\').split():os.environ[\'JAVA_TOOL_OPTIONS\']=(os.environ.get(\'JAVA_TOOL_OPTIONS\',\'\')+\' \'+option).strip()\n    from scienceworld import ScienceWorldEnv\n    from scienceworld.constants import JAR_PATH\n    dist=importlib.metadata.distribution(\'scienceworld\')\n    assert json.loads(dist.read_text(\'direct_url.json\') or \'{}\').get(\'vcs_info\',{}).get(\'commit_id\')==COMMIT,\'Install pinned version.\'\n    assert sha(Path(JAR_PATH))==JAR,\'Unexpected simulator JAR; stop.\'\n    out=source.parent/(\'scienceworld_recovery_diagnostic_\'+datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%S%fZ\'));out.mkdir()\n    save(out/\'config.json\',{\'source\':str(source),\'source_config_sha256\':sha(source/\'config.json\'),\n        \'source_results_sha256\':sha(source/\'results.json\'),\'protocol\':\'targeted one-call inverse tests; optional wait1 duration control\',\n        \'full_state_coverage_verified\':False,\'final_labels_created\':0,\'clock_and_goal_metadata_not_part_of_exported_world_comparison\':True})\n    (out/\'runner.py\').write_bytes(Path(__file__).read_bytes())\n    records=json.loads((source/\'results.json\').read_text());rows=[];errors=[];env=None;started=time.monotonic()\n    try:\n        env=ScienceWorldEnv(envStepLimit=250)\n        for record in records:\n            cid=record[\'id\'];folder=out/cid;folder.mkdir()\n            try:\n                print(cid,record[\'task\'],record[\'candidate\'],flush=True)\n                archived=list(source.glob(\'task_*/\'+cid+\'/snapshots.json.gz\'));assert len(archived)==1\n                with gzip.open(archived[0],\'rt\') as f: saved=json.load(f)\n                def snap(done):\n                    return {\'tree\':env.getObjectTree(),\'look\':env.look(),\'inventory\':env.inventory(),\n                        \'valid_actions\':sorted(set(env.get_valid_action_object_combinations())),\n                        \'moves\':int(env.get_num_moves()),\'score\':float(env.lastStepScore),\n                        \'goal_progress\':env.get_goal_progress(),\'done\':bool(done)}\n                def replay(seq):\n                    env.load(record[\'task\'],record[\'variation\'],\'\',generateGoldPath=False);env.reset();done=False;trace=[]\n                    for a in seq:\n                        if done:raise RuntimeError(\'Replay would continue past terminal state.\')\n                        obs,reward,done,info=env.step(a)\n                        trace.append({\'action\':a,\'observation\':obs,\'reward\':reward,\'done\':bool(done),\'moves\':int(env.get_num_moves())})\n                    return snap(done),trace\n                prefix=record[\'prefix\'];action=record[\'candidate\']\n                before,_=replay(prefix);after,trace=replay(prefix+[action])\n                assert before==saved[\'before\'] and after==saved[\'after\'],\'Current replay differs from archived snapshots; stop this case.\'\n                # Bindings are inspected at the pre-action state, where the action is available.\n                replay(prefix)\n                bindings=[x for x in env.get_valid_action_object_combinations_with_templates() if x.get(\'action\')==action]\n                ambiguity={\'bindings\':bindings,\'distinct_object_id_lists\':list({json.dumps(x.get(\'obj_ids\')) for x in bindings}),\n                           \'same_displayed_pour_names\':action.startswith(\'pour \') and len(action[5:].split(\' into \'))==2 and action[5:].split(\' into \')[0]==action[5:].split(\' into \')[1]}\n                save(folder/\'action_bindings.json\',ambiguity)\n                save(folder/\'candidate_changes.json\',comparison(before,after))\n                old_search=json.loads((archived[0].parent/\'search.json\').read_text()) if (archived[0].parent/\'search.json\').exists() else {}\n                depths=Counter(len(n[\'undo\']) for n in old_search.get(\'nodes\',[]))\n                branching=len(after[\'valid_actions\'])\n                coverage={\'prior_nodes_by_depth\':dict(depths),\'prior_queued_remaining\':old_search.get(\'queued_remaining\'),\n                    \'initial_recovery_branching\':branching,\'constant_branching_size_illustration_0_to_3\':sum(branching**h for h in range(4)),\n                    \'illustration_is_not_a_bound_or_measured_total\':True,\n                    \'prior_tested_one_step_fraction\':depths[1]/branching if branching else None}\n                save(folder/\'search_coverage.json\',coverage)\n                tests=[]\n                for undo in suggestions(action,before,after):\n                    end,endtrace=replay(prefix+[action]+undo);repeat,repeattrace=replay(prefix+[action]+undo)\n                    stable=end==repeat and endtrace==repeattrace\n                    comp=comparison(before,end)\n                    test={\'undo\':undo,\'replay_stable\':stable,\'comparison\':comp,\'elapsed_moves\':end[\'moves\']-before[\'moves\'],\'trace\':endtrace}\n                    # A measured duration control, never a replacement for literal restoration.\n                    if \'wait1\' in before[\'valid_actions\'] and not before[\'done\']:\n                        control,controltrace=replay(prefix+[\'wait1\'])\n                        duration=control[\'moves\']-before[\'moves\']\n                        test[\'wait_control\']={\'actions\':[\'wait1\'],\'elapsed_moves\':duration,\'duration_matches\':duration==test[\'elapsed_moves\'],\n                            \'time_relative_exported_world_match\':world(control)==world(end) if duration==test[\'elapsed_moves\'] else None,\n                            \'not_original_state_restoration\':True,\'trace\':controltrace}\n                    with gzip.open(folder/(\'targeted_\'+str(len(tests))+\'.json.gz\'),\'wt\') as f:json.dump({\'test\':test,\'end\':end,\'repeat_end\':repeat},f)\n                    tests.append({\'undo\':undo,\'replay_stable\':stable,\'exact_exported_world_match\':comp[\'exact_exported_world_match\'],\n                        \'temperature_field_differences\':comp[\'temperature_field_differences\'],\'other_difference_count\':comp[\'other_difference_count\'],\n                        \'wait_control\':test.get(\'wait_control\')})\n                row={\'id\':cid,\'candidate\':action,\'archive_replay_matches\':True,\'tests\':tests,\n                     \'no_inverse_tested\':not tests,\'same_displayed_pour_names\':ambiguity[\'same_displayed_pour_names\'],\n                     \'search_coverage\':coverage,\'final_label\':None}\n                rows.append(row);save(folder/\'summary.json\',row)\n                print(\'  targeted recoveries:\',len(tests),\'| exact exported matches:\',sum(t[\'exact_exported_world_match\'] and t[\'replay_stable\'] for t in tests),flush=True)\n            except Exception as exc:\n                errors.append({\'id\':cid,\'error\':str(exc)});(folder/\'error.txt\').write_text(traceback.format_exc())\n                print(\'  Case error saved:\',str(exc),flush=True)\n            save(out/\'progress.json\',{\'rows\':rows,\'errors\':errors})\n    finally:\n        tests=[t for r in rows for t in r[\'tests\']]\n        summary={\'cases_checked\':len(rows),\'targeted_recoveries_tested\':len(tests),\n            \'stable_exact_exported_world_witnesses\':sum(t[\'replay_stable\'] and t[\'exact_exported_world_match\'] for t in tests),\n            \'recoveries_with_only_temperature_differences\':sum(t[\'temperature_field_differences\']>0 and t[\'other_difference_count\']==0 for t in tests),\n            \'ambiguous_display_pour_cases\':[r[\'id\'] for r in rows if r[\'same_displayed_pour_names\']],\n            \'cases_without_tested_inverse\':[r[\'id\'] for r in rows if r[\'no_inverse_tested\']],\n            \'errors\':errors,\'elapsed_minutes\':round((time.monotonic()-started)/60,2),\n            \'final_labels_created\':0,\'full_state_coverage_verified\':False,\'scienceworld_requirement_met\':False,\'saved_to\':str(out)}\n        save(out/\'summary.json\',summary);save(out/\'results.json\',rows)\n        if env is not None:\n            try:env.close()\n            except Exception:pass\n    print(\'TARGETED RECOVERY DIAGNOSTIC COMPLETE\');print(json.dumps(summary,indent=2))\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--source\',default=\'/content/drive/MyDrive/ADNN_Irreversibility/results/\'+SOURCE)\n    main(p.parse_args())\n')
print("Diagnostic script ready.")


In [ ]:
import runpy, sys
old_argv = sys.argv[:]
try:
    sys.argv = ['/content/38_scienceworld_recovery_diagnostic_colab.py']
    runpy.run_path(sys.argv[0], run_name='__main__')
finally:
    sys.argv = old_argv


## Send back the final summary
Copy TARGETED RECOVERY DIAGNOSTIC COMPLETE and its JSON output into the project chat. Include any errors. No GPU or Hugging Face token is needed. Do not generate final labels from these diagnostic results.
